# Cache Transfer Workflow

This notebook demonstrates a common collaborative workflow using `CacheStack` and `transfer()`:

1. A **shared global cache** holds approved, vetted results (read-only for regular users).
2. Each user runs under a **local cache** stacked on top of the global one.
   - Cache hits in the global cache are served transparently.
   - New computations land only in the local cache.
3. After reviewing their local results, the user selects which ones to **promote** to the global cache via `transfer()`.

## 1. Setup

We create two caches:

- `_global_cache_rw` — a persistent SQL + PickleFile cache simulating a shared remote store (admin access only)
- `global_cache` — a read-only wrapper that regular users see (writes rejected), created via `.readonly()`
- `local_cache` — a fast in-memory cache for the user's current session
- `user_cache` — local stacked on top of global via `.push()`, so local results take priority

In [1]:
import tempfile
import time

from fleche import fleche, cache
from fleche.caches import BaseCache
from fleche.call import QueryCall

The admin-accessible cache uses persistent SQL + PickleFile storage.

In [2]:
tmp_dir = tempfile.TemporaryDirectory()

_global_cache_rw = BaseCache.from_config({
    "template": "sql", "root": tmp_dir.name, "values": "cloudpickle",
})
_global_cache_rw

Cache(values=ValuePickleFile(root=PosixPath('/tmp/tmpr7ce4vrv/values'), secret_key=(), compress=False, remaining_depth=1), calls=Sql(url='sqlite:////tmp/tmpr7ce4vrv/calls.db', echo=False))

Regular users get a read-only view via `.readonly()`, and a per-session in-memory cache stacked on top via `.push()`.

In [3]:
global_cache = _global_cache_rw.readonly()
local_cache = BaseCache.from_config({"template": "memory"})
user_cache = global_cache.push(local_cache)
user_cache

CacheStack(stack=(Cache(values=ValueMemory(storage={}, remaining_depth=1), calls=CallMemory(storage={})), ReadOnlyCache(cache=Cache(values=ValuePickleFile(root=PosixPath('/tmp/tmpr7ce4vrv/values'), secret_key=(), compress=False, remaining_depth=1), calls=Sql(url='sqlite:////tmp/tmpr7ce4vrv/calls.db', echo=False)))))

## 2. Define Functions

Two `@fleche`-decorated functions simulate a heavy computation and a post-processing step.

In [4]:
@fleche
def simulate(param: float) -> dict:
    """Expensive simulation — takes time, results worth sharing."""
    print(f"  [simulate] Running simulation for param={param}...")
    time.sleep(0.05)  # pretend this is expensive
    return {"param": param, "result": param ** 2 + 1.0}


@fleche
def postprocess(data: dict) -> float:
    """Quick post-processing — user-specific, not worth sharing."""
    print(f"  [postprocess] Processing {data}...")
    return data["result"] * 2.0

## 3. Seed the Global Cache (Admin Step)

An admin pre-populates the global cache with approved baseline results.
Regular users never do this — they only read from `global_cache` (the read-only view).

In [5]:
with cache(_global_cache_rw):
    for p in [1.0, 2.0, 3.0]:
        simulate(p)

  [simulate] Running simulation for param=1.0...
  [simulate] Running simulation for param=2.0...
  [simulate] Running simulation for param=3.0...


In [6]:
_global_cache_rw.table()

,name,module,timestart,timestop,walltime
55c4,simulate,__main__,2026-08-12 19:50:35.369092703+00:00,2026-08-12 19:50:35.419446230+00:00,0.050354
af0f,simulate,__main__,2026-08-12 19:50:35.315497637+00:00,2026-08-12 19:50:35.365865231+00:00,0.050368
fef5,simulate,__main__,2026-08-12 19:50:35.258429289+00:00,2026-08-12 19:50:35.308952570+00:00,0.050523


## 4. User Session

The user runs under `user_cache`, a `CacheStack` with local priority over global.

- Params **already in the global cache** (1.0, 2.0, 3.0) → cache hits, no recomputation.
- **New params** (4.0, 5.0, 6.0) → computed and saved to `local_cache` only.
- Post-processing results also land in `local_cache`.

Params 1–3 are already in the global cache — no `[simulate]` output means they are cache hits.

In [7]:
with cache(user_cache):
    for p in [1.0, 2.0, 3.0]:
        postprocess(simulate(p))

  [postprocess] Processing {'param': 1.0, 'result': 2.0}...
  [postprocess] Processing {'param': 2.0, 'result': 5.0}...
  [postprocess] Processing {'param': 3.0, 'result': 10.0}...


Params 4–6 are new — they will be computed and stored in `local_cache` only.

In [8]:
with cache(user_cache):
    for p in [4.0, 5.0, 6.0]:
        postprocess(simulate(p))

  [simulate] Running simulation for param=4.0...
  [postprocess] Processing {'param': 4.0, 'result': 17.0}...
  [simulate] Running simulation for param=5.0...
  [postprocess] Processing {'param': 5.0, 'result': 26.0}...
  [simulate] Running simulation for param=6.0...
  [postprocess] Processing {'param': 6.0, 'result': 37.0}...


## 5. Inspect the Caches

After the session:
- The **global cache** is unchanged (still only has params 1–3).
- The **local cache** has all new results (simulate + postprocess for params 4–6, and postprocess for 1–3 which were hits from global).

In [9]:
_global_cache_rw.table()

,name,module,timestart,timestop,walltime
55c4,simulate,__main__,2026-08-12 19:50:35.369092703+00:00,2026-08-12 19:50:35.419446230+00:00,0.050354
af0f,simulate,__main__,2026-08-12 19:50:35.315497637+00:00,2026-08-12 19:50:35.365865231+00:00,0.050368
fef5,simulate,__main__,2026-08-12 19:50:35.258429289+00:00,2026-08-12 19:50:35.308952570+00:00,0.050523


In [10]:
local_cache.table()

,name,module,timestart,timestop,walltime
fef5,simulate,__main__,2026-08-12 19:50:35.258429289+00:00,2026-08-12 19:50:35.308952570+00:00,0.050523
a7e7,postprocess,__main__,2026-08-12 19:50:35.453360081+00:00,2026-08-12 19:50:35.453533411+00:00,0.000173
af0f,simulate,__main__,2026-08-12 19:50:35.315497637+00:00,2026-08-12 19:50:35.365865231+00:00,0.050368
5c60,postprocess,__main__,2026-08-12 19:50:35.456634521+00:00,2026-08-12 19:50:35.456753492+00:00,0.000119
55c4,simulate,__main__,2026-08-12 19:50:35.369092703+00:00,2026-08-12 19:50:35.419446230+00:00,0.050354
127c,postprocess,__main__,2026-08-12 19:50:35.459400654+00:00,2026-08-12 19:50:35.459538698+00:00,0.000138
a1d5,simulate,__main__,2026-08-12 19:50:35.465635061+00:00,2026-08-12 19:50:35.515836716+00:00,0.050202
d75f,postprocess,__main__,2026-08-12 19:50:35.516886711+00:00,2026-08-12 19:50:35.517035007+00:00,0.000148
b810,simulate,__main__,2026-08-12 19:50:35.517865419+00:00,2026-08-12 19:50:35.568025827+00:00,0.050160
3c2c,postprocess,__main__,2026-08-12 19:50:35.569146395+00:00,2026-08-12 19:50:35.569267511+00:00,0.000121


## 6. Filter Before Transfer

The user reviews their local cache and decides only the `simulate` results are worth promoting to the global cache.
Post-processing results are user-specific and stay local.

`filter()` returns a `FilteredCache` — a read-only view that `transfer()` will iterate over.

In [11]:
simulations_only = local_cache.filter(QueryCall.from_call(simulate, param=None))
simulations_only.table()

,name,module,timestart,timestop,walltime
fef5,simulate,__main__,2026-08-12 19:50:35.258429289+00:00,2026-08-12 19:50:35.308952570+00:00,0.050523
af0f,simulate,__main__,2026-08-12 19:50:35.315497637+00:00,2026-08-12 19:50:35.365865231+00:00,0.050368
55c4,simulate,__main__,2026-08-12 19:50:35.369092703+00:00,2026-08-12 19:50:35.419446230+00:00,0.050354
a1d5,simulate,__main__,2026-08-12 19:50:35.465635061+00:00,2026-08-12 19:50:35.515836716+00:00,0.050202
b810,simulate,__main__,2026-08-12 19:50:35.517865419+00:00,2026-08-12 19:50:35.568025827+00:00,0.050160
1e5f,simulate,__main__,2026-08-12 19:50:35.570091009+00:00,2026-08-12 19:50:35.620252609+00:00,0.050162


## 7. Transfer to the Global Cache

The user (or admin) calls `transfer()` to promote the selected results into the global cache.
With the default `overwrite=False`, entries already present in the global cache are left untouched.
After this, any user will get cache hits for `simulate(4.0)`, `simulate(5.0)`, and `simulate(6.0)` without recomputation.

In [12]:
simulations_only.transfer(_global_cache_rw)

Not transferring fef51eb9e42e5b192ed1ace6f10e5396fb544e880cec69fd2622324666735583: already exists in target and overwrite=False


Not transferring af0fdbd1f495ffdd36b0921657f80d9fe6f78d662877256b6101d02b2d5703d3: already exists in target and overwrite=False


Not transferring 55c426f0f3a90df4dc2416511caf1479221753362100eb143261ba28c98e5125: already exists in target and overwrite=False


In [13]:
_global_cache_rw.table()

,name,module,timestart,timestop,walltime
1e5f,simulate,__main__,2026-08-12 19:50:35.570091009+00:00,2026-08-12 19:50:35.620252609+00:00,0.050162
55c4,simulate,__main__,2026-08-12 19:50:35.369092703+00:00,2026-08-12 19:50:35.419446230+00:00,0.050354
a1d5,simulate,__main__,2026-08-12 19:50:35.465635061+00:00,2026-08-12 19:50:35.515836716+00:00,0.050202
af0f,simulate,__main__,2026-08-12 19:50:35.315497637+00:00,2026-08-12 19:50:35.365865231+00:00,0.050368
b810,simulate,__main__,2026-08-12 19:50:35.517865419+00:00,2026-08-12 19:50:35.568025827+00:00,0.050160
fef5,simulate,__main__,2026-08-12 19:50:35.258429289+00:00,2026-08-12 19:50:35.308952570+00:00,0.050523


## 8. Verify: Second User Gets Cache Hits

A second user starts a fresh session with their own empty local cache.
They can now load `simulate(4.0)` – `simulate(6.0)` without any computation.

In [14]:
local_cache_2 = BaseCache.from_config({"template": "memory"})
user_cache_2 = global_cache.push(local_cache_2)

All six params should now be cache hits — no `[simulate]` output expected.

In [15]:
with cache(user_cache_2):
    for p in [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]:
        simulate(p)

In [16]:
tmp_dir.cleanup()